# ✅ Solución — Reto del Cargamento Sospechoso
### Solo para el equipo docente (clave de corrección)

# 📦 Reto: Inspección del Cargamento Sospechoso
### Bloque 3 — Reto de aplicación (20 min)

Sois Inspectores Vogon en formación. Ha llegado un cargamento a la nave y tenéis que construir el modelo Pydantic que lo valide **antes** de dejarlo pasar. Trabajad en parejas y completad los `# TODO`.

**Recordatorio de herramientas que acabáis de ver:**
- `class Modelo(BaseModel): campo: tipo`
- `Field(..., gt=..., le=..., min_length=..., max_length=...)`
- `Literal["a", "b", "c"]` para valores cerrados
- `@field_validator("campo")` para reglas personalizadas
- Modelos anidados: un campo puede ser otro `BaseModel`

In [1]:
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError
from typing import Literal, Optional


## Paso 1 — Modelo `Remitente`

Todo cargamento tiene un remitente. Completad este modelo con:
- `nombre_empresa: str` (entre 2 y 60 caracteres)
- `planeta: str`
- `codigo_licencia: str` — debe tener **exactamente 6 caracteres** (usad `min_length` y `max_length` iguales a 6)

## Paso 2 — Modelo `CargamentoSospechoso`

Ahora el modelo principal, con estos campos:
- `codigo_contenedor: str`
- `peso_kg: float` — debe ser mayor que 0 y menor o igual a 500
- `tipo_contenido: Literal["alimentos", "tecnologia", "papeleo", "desconocido"]`
- `remitente: Remitente` (¡modelo anidado, usad el que hicisteis arriba!)
- `nivel_riesgo: int` — opcional, por defecto `1` (usad `Optional[int] = 1` o `Field(1, ...)`)

## Paso 3 — Validador personalizado

Añadid un `@field_validator` sobre `codigo_contenedor` que obligue a que **siempre empiece por el prefijo `"VGN-"`** (código de aduana Vogona). Si no cumple, debe lanzar un `ValueError` con un mensaje claro.

Pista: podéis reescribir la clase completa aquí abajo, copiando lo que hicisteis en el Paso 2 y añadiendo el validador.


In [ ]:
class Remitente(BaseModel):
    nombre_empresa: str = Field(..., min_length=2, max_length=60)
    planeta: str
    codigo_licencia: str = Field(..., min_length=6, max_length=6)


class CargamentoSospechoso(BaseModel):
    codigo_contenedor: str
    peso_kg: float = Field(..., gt=0, le=500)
    tipo_contenido: Literal["alimentos", "tecnologia", "papeleo", "desconocido"]
    remitente: Remitente
    nivel_riesgo: Optional[int] = 1

    @field_validator("codigo_contenedor") # Valida un campo asilado
    @classmethod
    def debe_empezar_por_vgn(cls, valor: str) -> str:
        if not valor.startswith("VGN-"):
            raise ValueError("el código de contenedor debe empezar por 'VGN-'")
        return valor

    # Bonus: model_validator para reglas que dependen de varios campos a la vez
    @model_validator(mode="after")
    # validador que se ejecuta cuando el objeto entero ya está construido, no campo por campo. 
    # Es distinto de @field_validator, que solo puede mirar un campo aislado.
    def riesgo_minimo_si_desconocido(self):
        if self.tipo_contenido == "desconocido" and self.nivel_riesgo < 5:
            raise ValueError("un cargamento de tipo 'desconocido' requiere nivel_riesgo >= 5")
        return self


## Paso 4 — Probad vuestro modelo

Descomentad y ejecutad estas pruebas. **Las tres deben comportarse como se indica en el comentario.**


In [3]:
# Prueba 1: debe funcionar
cargamento_ok = CargamentoSospechoso(
    codigo_contenedor="VGN-4471",
    peso_kg=120.5,
    tipo_contenido="tecnologia",
    remitente={"nombre_empresa": "Sirius Cybernetics", "planeta": "Sirio", "codigo_licencia": "AB12CD"},
)
print(cargamento_ok)


codigo_contenedor='VGN-4471' peso_kg=120.5 tipo_contenido='tecnologia' remitente=Remitente(nombre_empresa='Sirius Cybernetics', planeta='Sirio', codigo_licencia='AB12CD') nivel_riesgo=1


In [4]:
# Prueba 2: debe fallar (prefijo incorrecto)
try:
    cargamento_mal = CargamentoSospechoso(
        codigo_contenedor="XX-0001",
        peso_kg=50,
        tipo_contenido="papeleo",
        remitente={"nombre_empresa": "Anon", "planeta": "?", "codigo_licencia": "ZZ9999"},
    )
except ValidationError as e:
    print(e)


1 validation error for CargamentoSospechoso
codigo_contenedor
  Value error, el código de contenedor debe empezar por 'VGN-' [type=value_error, input_value='XX-0001', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [5]:
# Prueba 3: debe fallar (peso negativo)
try:
    cargamento_mal2 = CargamentoSospechoso(
        codigo_contenedor="VGN-0002",
        peso_kg=-5,
        tipo_contenido="alimentos",
        remitente={"nombre_empresa": "Anon", "planeta": "?", "codigo_licencia": "ZZ9999"},
    )
except ValidationError as e:
    print(e)


1 validation error for CargamentoSospechoso
peso_kg
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


In [6]:
# Prueba bonus: tipo "desconocido" con riesgo bajo debe fallar
try:
    cargamento_bonus = CargamentoSospechoso(
        codigo_contenedor="VGN-9999",
        peso_kg=10,
        tipo_contenido="desconocido",
        remitente={"nombre_empresa": "Anon", "planeta": "?", "codigo_licencia": "ZZ9999"},
        nivel_riesgo=2,
    )
except ValidationError as e:
    print(e)


1 validation error for CargamentoSospechoso
  Value error, un cargamento de tipo 'desconocido' requiere nivel_riesgo >= 5 [type=value_error, input_value={'codigo_contenedor': 'VG...99'}, 'nivel_riesgo': 2}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
